# Full Validation Run Report

This notebook runs the full validation pipeline using the current project configuration, keeping the configured repetition count from `config/base.yaml`.

Behavior:
- uses the processed CSV configured in `config/paths.yaml`
- uses the validation settings from `config/validation/validation.yaml`
- forces `validation_bool: true` through a temporary runtime config
- preserves the current `validation_repetitions` value from `config/base.yaml`
- runs the full validation loop end to end
- loads the JSON summary and visualizes the results

This is expected to take a long time. The main execution cell is intentionally a full run.

In [1]:
from __future__ import annotations

import json
import sys
import tempfile
from pathlib import Path
from textwrap import dedent

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from pipeline import Pipeline
from validation import ValidationPipeline

In [2]:
def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(dedent(text).lstrip(), encoding="utf-8")


def build_runtime_base_config(project_root: Path) -> tuple[Path, tempfile.TemporaryDirectory]:
    base_pipeline = Pipeline(project_root / "config" / "base.yaml")
    base_config = base_pipeline.root_config
    repetitions = int(base_config.get("validation_repetitions", 1))
    validation_config_path = (project_root / "config" / "validation" / "validation.yaml").resolve()
    paths_config_path = (project_root / "config" / "paths.yaml").resolve()
    recommendation_config_path = (project_root / "config" / "recommendation" / "pipeline.yaml").resolve()

    temp_dir = tempfile.TemporaryDirectory()
    runtime_base_path = Path(temp_dir.name) / "base_runtime_validation.yaml"
    write_text(
        runtime_base_path,
        f"""
        project:
          name: {base_config['project']['name']}
          seed: {base_config['project'].get('seed', 42)}

        validation_bool: true
        validation_repetitions: {repetitions}

        paths:
          config_path: {paths_config_path}

        pipeline:
          config_path: {recommendation_config_path}

        validation:
          config_path: {validation_config_path}
        """,
    )
    return runtime_base_path, temp_dir


In [3]:
base_pipeline = Pipeline(PROJECT_ROOT / "config" / "base.yaml")
validation_pipeline_preview = ValidationPipeline(PROJECT_ROOT / "config" / "base.yaml")

print("Current base config:")
print(json.dumps(base_pipeline.root_config, indent=2, default=str))
print()
print("Resolved validation config preview:")
print(json.dumps({
    "repetitions": validation_pipeline_preview.config.repetitions,
    "seed": validation_pipeline_preview.config.seed,
    "users_per_repetition": validation_pipeline_preview.config.users_per_repetition,
    "recommendation_k": validation_pipeline_preview.config.recommendation_k,
    "metrics_k": validation_pipeline_preview.config.metrics_k,
    "source_csv_path": str(validation_pipeline_preview.config.source_csv_path),
    "recommendation_pipeline_config_path": str(validation_pipeline_preview.config.recommendation_pipeline_config_path),
    "summary_path": str(validation_pipeline_preview.config.summary_path),
}, indent=2))

Current base config:
{
  "project": {
    "name": "classic_methods",
    "seed": 42
  },
  "validation_bool": false,
  "validation_repetitions": 50,
  "paths": {
    "config_path": "config/paths.yaml"
  },
  "pipeline": {
    "config_path": "config/recommendation/pipeline.yaml"
  },
  "validation": {
    "config_path": "config/validation/validation.yaml"
  }
}

Resolved validation config preview:
{
  "repetitions": 50,
  "seed": 42,
  "users_per_repetition": 20,
  "recommendation_k": 10,
  "metrics_k": 10,
  "source_csv_path": "C:\\Users\\lovro\\Desktop\\hackatoni\\LUMEN_DS_processed.csv",
  "recommendation_pipeline_config_path": "C:\\Users\\lovro\\Desktop\\hackatoni\\LUMEN2026\\classic_methods\\config\\recommendation\\pipeline_validation.yaml",
  "summary_path": "C:\\Users\\lovro\\Desktop\\hackatoni\\LUMEN2026\\classic_methods\\output\\validation_summary.json"
}


In [4]:
runtime_base_path, runtime_dir = build_runtime_base_config(PROJECT_ROOT)
validation_pipeline = ValidationPipeline(runtime_base_path)

summary = validation_pipeline.run()
summary_path = validation_pipeline.config.summary_path

print(f"Validation summary written to: {summary_path}")
print(f"Repetitions run: {summary['repetitions_run']}")
print(f"Users per repetition: {summary['users_per_repetition']}")
print(f"Metrics k: {summary['metrics_k']}")

KeyError: 'Unknown config reference: paths.processed_data_csv'

In [ ]:
summary_path = validation_pipeline.config.summary_path
summary = json.loads(summary_path.read_text(encoding="utf-8"))

repetition_df = pd.DataFrame(summary["repetition_results"])
metrics_df = pd.DataFrame(summary["metrics"]).T.reset_index().rename(columns={"index": "metric"})
user_counts_df = pd.DataFrame(
    [
        {"CustomerID": int(user_id), "times_tested": count}
        for user_id, count in summary["per_user_test_counts"].items()
    ]
).sort_values(["times_tested", "CustomerID"], ascending=[False, True], kind="stable")

metrics_df

In [ ]:
print("Aggregate metrics")
for metric_name, metric_values in summary["metrics"].items():
    print(f"{metric_name}: mean={metric_values['mean']:.6f}, std={metric_values['std']:.6f}")

print()
print(f"Requested repetitions: {summary['requested_repetitions']}")
print(f"Repetitions run: {summary['repetitions_run']}")
print(f"Users per repetition: {summary['users_per_repetition']}")
print(f"Recommendation k: {summary['recommendation_k']}")
print(f"Metrics k: {summary['metrics_k']}")
print(f"Summary path: {summary_path}")

print()
print("Top users by number of evaluations")
display(user_counts_df.head(20))

print("Per-repetition metrics preview")
display(repetition_df[["repetition", "evaluated_user_count", "ndcg_at_k", "recall_at_k", "mrr_at_k"]].head(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(repetition_df["repetition"], repetition_df["ndcg_at_k"], marker="o", linewidth=1.5)
axes[0, 0].set_title("NDCG@K by Repetition")
axes[0, 0].set_xlabel("Repetition")
axes[0, 0].set_ylabel("NDCG@K")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(repetition_df["repetition"], repetition_df["recall_at_k"], marker="o", linewidth=1.5, color="tab:orange")
axes[0, 1].set_title("Recall@K by Repetition")
axes[0, 1].set_xlabel("Repetition")
axes[0, 1].set_ylabel("Recall@K")
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(repetition_df["repetition"], repetition_df["mrr_at_k"], marker="o", linewidth=1.5, color="tab:green")
axes[1, 0].set_title("MRR@K by Repetition")
axes[1, 0].set_xlabel("Repetition")
axes[1, 0].set_ylabel("MRR@K")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].hist(user_counts_df["times_tested"], bins=range(1, int(user_counts_df["times_tested"].max()) + 2), color="tab:purple", align="left", rwidth=0.85)
axes[1, 1].set_title("Distribution of User Test Counts")
axes[1, 1].set_xlabel("Times Tested")
axes[1, 1].set_ylabel("Number of Users")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(metrics_df["metric"], metrics_df["mean"], yerr=metrics_df["std"], capsize=6, color=["tab:blue", "tab:orange", "tab:green"])
ax.set_title("Aggregate Validation Metrics")
ax.set_ylabel("Mean Score")
ax.grid(axis="y", alpha=0.3)
plt.show()